## Stage 00 - sentencize both corpora

One cascade, both corpora, in the order the text was actually produced: **the page layout
decides the units, and the trained segmenter runs last, inside one unit at a time**.

1. **Normalise** - mojibake repair, NFKC, missing space after a full stop.
2. **Strip page furniture** - a line, or a line-opening, the document repeats three or more
   times is a running head, a footer or a section banner. Extraction pastes such a banner
   into the middle of a line as often as it leaves it on a line of its own, so it is removed
   everywhere it occurs.
3. **Unwrap** - close up the lines the layout broke, keep the ones the author broke. A line
   is closed up when the next one opens in lower case, when it ends on a one- or two-letter
   word, or when it runs to the document's own full measure. A line ending in `.!?;:` is a
   unit boundary and is never closed up.
4. **Break the itemized lists** - enumerators, bullets, semicolon items, and a run of
   capitals extraction ran into the paragraph beside it.
5. **Distribute the colon header** - onto the run of items under it too short to carry a
   topic alone, never onto a full proposal.
6. **Segment** - a unit with a full stop in its interior goes to the trained segmenter,
   which decides which of those full stops end a sentence. A full stop closing an
   abbreviation is not one of them: the abbreviation list completes the tokenizer's own,
   which is thin for Portuguese, and a boundary after an abbreviation that requires a
   following word is refused whatever the model proposes.
7. **Reattach stranded tails**, then force a floor of `MIN_WORDS` words per chunk: a bare
   section-header label, a colon header `fix_colon_lists` missed, a leaked table cell, or a
   punctuation-only fragment the segmenter carved out on its own (a lone `.`, `...`, a stray
   list marker) is merged into a neighbour rather than written out as its own chunk. Then
   cap at 150 words with a 10-word stride.

Output: `chunk_id, doc_id, party_label, chunk_text, word_count` plus the corpus metadata,
written as feather and csv.


## 1. Mount Drive + install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# spaCy + the two trained models + ftfy. Pinned to match the local build (spaCy 3.8.x, models 3.8.0).
!pip -q install "spacy>=3.8,<3.9" ftfy
!python -m spacy download en_core_web_lg
!python -m spacy download pt_core_news_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 715.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 2.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Config

In [ ]:
import os
from pathlib import Path

# Drive root (adjust if your Drive mounts the folder elsewhere).
ROOT = Path(os.getenv('TOPIC2IRT_ROOT', '/content/drive/MyDrive/Papers/transfer_learning/topic2irt'))
assert ROOT.exists(), f"ROOT not found: {ROOT}  (check the Drive path)"

# ---- tunables ----
N_PROCESS      = os.cpu_count()  # use ALL available Colab cores for the segmenter pass
MAX_WORDS      = 150        # hard cap: longer chunks are brute-force window-split
STRIDE_OVERLAP = 10         # words of overlap between consecutive windows
SHORT_MAX      = 4          # a chunk <= this many words can be a stranded tail
MIN_WORDS      = 5          # hard floor: every written chunk is merged up to at least this
HEADER_MAX     = 12         # a line <= this many words ending ":" is a list header
NEEDY_MAX      = 8          # a unit this short takes the colon header above it as context
FURNITURE_MIN_REPEAT = 3    # a line or a line-opening repeated this often is page furniture
FURNITURE_MIN_PREFIX = 5    # ... and a repeated line-opening must run to this many words
FURNITURE_MAX_WORDS  = 30
SENTER_MAX_LEN = 3_000_000
print("N_PROCESS =", N_PROCESS, "| ROOT ok")


## 3. Pipeline (normalize · furniture · unwrap · list cascade · segmenter · cap)

In [ ]:
import csv, re, sys, unicodedata
from collections import Counter, OrderedDict
import ftfy, pandas as pd, spacy
from tqdm.auto import tqdm

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

# uppercase incl. Portuguese accents; lowercase/digit incl. accents.
_UP = "A-ZÁÉÍÓÚÀÃÂÊÎÔÛÇÑÜ"
_LO = "a-z0-9áéíóúàãâêîôûçñü"

_MISSING_SPACE_RE = re.compile(rf"(?<=[{_LO}])([.!?])(?=[{_UP}])")
_CONT_START_RE = re.compile(rf"^[^\w]*[{_LO}]")
# closes on a one- or two-letter word ("de", "do", "e", "of", "to"): the page cut it there
_DANGLING_RE = re.compile(r"(?:^|\s)[^\W\d_]{1,2}$")
_UNIT_END = ".!?;:"
_MIN_LINES, _MIN_MEASURE, _MEASURE_SLACK = 10, 40, 3


# ------------------------------------------------------------------ abbreviations
# A full stop closing an abbreviation is not a sentence boundary.  The trained segmenter
# infers this from its tokenizer's exception list, which is thin for Portuguese, so the
# list is completed here and enforced independently of the model.
#
# Only abbreviations that REQUIRE a following word are listed, because binding the full
# stop into the token also takes the choice away from the model.  An abbreviation that can
# close a sentence is therefore left alone entirely: "etc." ends one in 216 of its 473 US
# occurrences, and guarding it welded those sentences to the next.  Same for "Inc.", "Co.",
# "Ltda.", "Cia." and the name suffix "Jr.", all of which the model already handles.
#
# Membership was decided by reading, one at a time, every form of six letters or fewer that
# appears with a trailing period at least twice in either corpus: 8,538 Portuguese and 3,933
# English.  Six is the longest entry in the Moses nonbreaking-prefix lists ("Messrs"), which
# the reading was then unioned with.  Forms below that frequency are 0.44% of Portuguese and
# 0.53% of English period tokens and are not covered.
#
# Two classes are deliberately absent.  Single letters and Roman numerals are in the Moses
# lists but are enumerators here far more often than initials ("II." opens a section 1,542
# times, "XXI." closes a line 103 times), so refusing a boundary after them would weld list
# items together.  Forms that collide with a word are dropped on the corpus evidence that
# convicted them: "mar." is the sea closing a sentence 105 times against 15 as the month,
# "km." closes 38, "op." would be Orcamento Participativo closing 69, "par." is the noun.
#
# BOUND_ANY holds forms safe in any case.  BOUND_CASED holds forms safe only as written,
# because their upper-case spelling is a different word in this corpus: PE is Pernambuco,
# MS is Mato Grosso do Sul, CF is the Constitution, AP is Amapa, and each ends sentences.
_ABBREV_BOUND_ANY = {
    "pt": """
        Dr Dra Drs Dras Sr Sra Srs Sras Srta Srtas Prof Profa Profs Profas
        Exmo Exma Exmos Exmas Ilmo Ilma Sto Sta Cel Ten Sgt Cap Maj Gen Alm
        Dep Pref Pres Sen Eng Enga Engo Adm Adj Adv Cand Msgr Insp Supt Surg
        Pfc Pvt Sfc Comdr Mlle Mme MM Hosp Mons NSra
        Av Avs Rod Jd Trav Lg Pç Pça Pq Vl Lot Conj Pov Mun Munic Est Estr Apto
        Bl Qd Qt Cj Sl Bal Rib Res Con Tel
        Art Arts inc incs Cód Nr Id
        ed pag pág fls fl vol vols fig figs Org Orgs sec séc ref aprox
        cit col ord obs Ex ibid apud etal et eg Econ
        jan fev abr mai jun jul ago set out nov dez
        Sec Secr Exec Estab Gab Serv Nac Soc Min Coord Cood Ativ Transf Quant Qtd
        Tab Aux Esc Matr Tec Téc Unid Und Rec Alt Seq Pav Conv Part Ens Cond Nfe
        Lda Corp Rt Ass Assoc Assist Atend Cad Cons Const Cont Coop Dec Des Depto
        Dir Doc Enf Fed Freq Fund Gal Gov Ind Port Prev Prog Rev Seg
        Ton Arq Cx ag www
        """,
    "en": """
        Adj Adm Adv Ald Asst Assn Ave Bldg Blvd Brig Bros Capt Cmdr Col Comdr Con
        Cpl Dept Depts Dr Drs Ens Esq Fmr Ft Gen Gov Hon Hosp Hr Insp Lt MM Maj
        Messrs Mlle Mme Mr Mrs Msgr Mt Op Ord Pfc Ph Pres Prof Pub Pvt Rd Rep
        Reps Res Ret Rev Rt Sec sect Sen Sens Sfc Sgt Sr St Supt Surg Univ
        approx cf fig figs vol vols pp et esp ie Ed Prop Spc Stat Ste Sq
        avg Ch Cong www
        """,
}
# Safe only in the case written here.  The upper-case spelling of each of these is a
# different, sentence-closing token in the corpus: PE Pernambuco, MS Mato Grosso do Sul,
# CF the Constitution, AP Amapa, AL Alagoas, PG Ponta Grossa, ESP Espirito Santo, CMDR the
# rural development council, ID identification.  "Ver." is the councillor, never the verb,
# which is why it appears here and not above.
_ABBREV_BOUND_CASED = {
    "pt": "Pe Ms Cf Ap Ver al Pg Esp Cmdr",
    "en": "al Id",
}


def _abbrev_forms(any_case, cased):
    """The cases each abbreviation is actually written in, minus the unsafe ones."""
    out = set(cased.split())
    for w in any_case.split():
        out.update({w, w.lower(), w.upper(), w.lower().capitalize()})
    return out


ABBREV_BOUND = {k: _abbrev_forms(v, _ABBREV_BOUND_CASED[k])
                for k, v in _ABBREV_BOUND_ANY.items()}
# The tokenizer is given exactly what the guard refuses, and nothing more.
ABBREV_ALL = ABBREV_BOUND


# ------------------------------------------------------------------ page furniture
FURNITURE_MIN_PREFIX = 5


def _furniture(lines):
    """Strings a document repeats are page furniture: running heads, footers, banners.

    Two sources.  A line that stands alone three or more times is a header or a footer.  A
    phrase that opens three or more lines is a section banner that extraction ran into the
    text under it; five words is the floor, so an ordinary shared opening ("I will fight
    for") is never mistaken for one.
    """
    stripped = [ln.strip() for ln in lines if ln.strip()]
    counts = Counter(stripped)
    furn = {s for s, n in counts.items()
            if n >= FURNITURE_MIN_REPEAT and 1 <= len(WORD_RE.findall(s)) <= FURNITURE_MAX_WORDS}
    pref = Counter()
    for s in stripped:
        w = s.split()
        for k in range(FURNITURE_MIN_PREFIX, min(len(w), FURNITURE_MAX_WORDS)):
            pref[" ".join(w[:k])] += 1
    for s, n in sorted(pref.items(), key=lambda kv: -len(kv[0])):
        # A banner is a complete unit, so it closes on a content word.  Candidates ending
        # in a short function word ("Reforma e ampliacao do", "CAPTAR RECURSOS PARA
        # CONSTRUCAO DE") are an opening several proposals share, not furniture, and
        # removing them would take the verb off every one of those proposals.
        if n < FURNITURE_MIN_REPEAT or len(s.split()[-1].strip(",;:.")) < 5:
            continue
        if not any(s in f for f in furn):
            furn.add(s)
    return furn


def strip_furniture(text):
    """Drop a running header, footer or section banner wherever it occurs.

    Extraction pastes such a banner into the middle of a line as often as it leaves it on
    a line of its own, which is what welds a heading onto an unrelated proposal.  Removing
    it everywhere also lets the wrap test below close up the sentence it interrupted.
    """
    lines = text.split("\n")
    furn = _furniture(lines)
    if not furn:
        return text
    pat = re.compile("|".join(re.escape(s) for s in sorted(furn, key=len, reverse=True)))
    out = []
    for ln in lines:
        if ln.strip() in furn:
            continue
        out.append(pat.sub("\n", ln) if pat.search(ln) else ln)
    return "\n".join(out)


# ------------------------------------------------------------------ line unwrapping
def _unwrap(text):
    """Close up the lines the page layout broke, keep the ones the author broke.

    Two layout-free tests decide.  The next line opening in lower case or with a digit
    makes the break a wrap.  So does a line running to the document's own full measure: an
    item that stops short of the measure stopped on purpose.  A line ending in ".!?;:" is a
    unit boundary and is never closed up, and a blank line always survives.
    """
    lines = text.split("\n")
    if len(lines) < _MIN_LINES:
        return text
    lens = sorted(len(ln.rstrip()) for ln in lines if ln.strip())
    if not lens:
        return text
    measure = lens[int(0.90 * (len(lens) - 1))]
    use_measure = measure >= _MIN_MEASURE
    out = [lines[0].rstrip()]
    last = len(out[0])                      # the measure applies to the source line, never
    for ln in lines[1:]:                    # to the accumulated one, which would run away
        cur, prev = ln.strip(), out[-1].rstrip()
        if (cur and prev and prev[-1] not in _UNIT_END
                and (_CONT_START_RE.match(cur)
                     or _DANGLING_RE.search(prev)
                     or (use_measure and last >= measure - _MEASURE_SLACK))):
            out[-1] = f"{prev} {cur}"
        else:
            out.append(ln.rstrip())
        last = len(ln.rstrip())
    return "\n".join(out)


def normalize_text(text):
    text = ftfy.fix_text(text)
    text = unicodedata.normalize("NFKC", text)
    text = _MISSING_SPACE_RE.sub(r"\1 ", text)
    return _unwrap(strip_furniture(text))


# ------------------------------------------------------------------ itemized lists
HORIZ_NOISE_RE = re.compile(r"[-–_=\.]{4,}")
REPEAT_RE = re.compile(r"(.{6,120})(?:\s+\1){3,}", re.IGNORECASE)
MULTI_NEWLINE_RE = re.compile(r"\n{2,}")
NEWLINE_NUMBERED_RE = re.compile(r"\n\s*\d+[\.\d]*[\.\)]?\s+")
NEWLINE_LETTERED_RE = re.compile(r"\n\s*[a-zA-Z]\)\s+")
NEWLINE_DASH_RE = re.compile(r"\n\s*[-–]\s+")
INLINE_NUMBERED_RE = re.compile(
    rf"(?<=\S)\s+\d+[\.\d]*[\.\)]?\s*[–-]\s+|\s+\d+[\.\d]*\.\s+(?=[{_UP}])")
LIST_MARKERS = r"☼|•|·|\*|➢|➤|►|▶|▸|→|✓|✔|●|○|◦|◆|■|□|▪|‣|⁃|⦁|∙|§|»|«|★|✰"
MARKER_SPLIT_RE = re.compile(rf"(?<!\A)\s*(?={LIST_MARKERS})")
LIST_LEAD_RE = re.compile(rf"^\s*(?:{LIST_MARKERS})")

# A semicolon opens a new item when what follows is capitalised or numbered.  A lower-case
# follower joins two clauses of one sentence, and so does the English pronoun "I".
SEMI_SPLIT_RE = re.compile(rf";\s+(?!I\s)(?=[^\w\s]{{0,3}}\s*[{_UP}0-9])")
PUNCT_ONLY_RE = re.compile(r"^[\W_]+$")
WORD_RE = re.compile(r"\b[\w'-]+\b")

_LEAD_ENUM_RE = re.compile(
    r"^\s*(?:\(?\d{1,3}(?:\.\d{1,3})*\)"
    r"|\(?[a-zA-Z]\)"
    r"|\d{1,3}(?:\.\d{1,3})*\s*[.)–-]"
    r"|[ivxIVX]{1,4}[.)])\s*")

# A run of capitals at the head of a unit, followed by ordinary text, is a heading that
# extraction ran into the paragraph below it.  The follower must be a letter: a digit is
# part of the heading ("IGARAPE-MIRI 100% LIMPA"), not the start of the text under it.
_LOW = "a-záéíóúàãâêîôûçñü"
_CAPS_W = rf"[{_UP}][{_UP}0-9'’\-\.º°]*"
CAPS_HEAD_RE = re.compile(
    rf"^\s*((?:{_CAPS_W}\s+){{1,10}})(?=[{_UP}][{_LOW}]|[{_LOW}])")


def _clean_repetition(text):
    return REPEAT_RE.sub(lambda m: m.group(1), text)


def _split_caps_head(frag):
    """Cut a leading all-capitals heading off the text extraction ran into it.

    A single capitalised word is an acronym as often as a heading, so it is cut only when
    it is long and the text under it opens in title case, which an acronym's continuation
    does not ("junto ao INCRA, buscando" stays whole, "SAUDE Medidas de prevencao" does not).
    The mirror rule, cutting a capitalised run off the END of a line, is deliberately absent:
    a plural acronym ("EPIs OS AGENTES COMUNITARIOS") is indistinguishable from a heading
    there, and it split far more units than it repaired.
    """
    m = CAPS_HEAD_RE.match(frag)
    if not m:
        return [frag]
    # a one-letter capital closing the run is an article the heading borrowed from the
    # sentence under it ("VICE PREFEITO ATUANTE O | Vice-prefeito ..."); give it back
    words, moved = m.group(1).split(), []
    while len(words) > 1 and len(words[-1].strip(".")) == 1:
        moved.insert(0, words.pop())
    head = " ".join(words)
    tail = " ".join(moved + [frag[m.end():].strip()]).strip()
    single_ok = (len(words) == 1 and len(words[0]) >= 5
                 and re.match(rf"[{_UP}][{_LOW}]", tail))
    if not (len(words) >= 2 or single_ok) or len(WORD_RE.findall(tail)) < 3:
        return [frag]
    return [head, tail]


def split_units(text):
    """Break one document into units, grouped by blank-line block.

    Returns a list of blocks, each a list of units.  A block is the run of lines between
    two blank lines, which is the span an itemized list occupies on the page; keeping the
    blocks apart is what bounds the colon header below.
    """
    text = _clean_repetition(HORIZ_NOISE_RE.sub(" ", text)).strip()
    if not text:
        return []
    blocks = []
    for block in MULTI_NEWLINE_RE.split(text):
        frags = NEWLINE_NUMBERED_RE.split(block)
        frags = [s for f in frags for s in INLINE_NUMBERED_RE.split(f)]
        frags = [s for f in frags for s in NEWLINE_LETTERED_RE.split(f)]
        frags = [s for f in frags for s in NEWLINE_DASH_RE.split(f)]
        frags = [s for f in frags for s in f.split("\n")]
        frags = [s for f in frags for s in MARKER_SPLIT_RE.split(f)]
        frags = [s for f in frags for s in SEMI_SPLIT_RE.split(f)]
        out = []
        for frag in frags:
            frag = _LEAD_ENUM_RE.sub("", re.sub(r"\s+", " ", frag).strip())
            frag = frag.rstrip("; ")
            if not frag or PUNCT_ONLY_RE.match(frag):
                continue
            out.extend(p for p in _split_caps_head(frag) if p)
        if out:
            blocks.append(out)
    return blocks


def fix_colon_lists(blocks, max_header_words=HEADER_MAX, needy_words=NEEDY_MAX):
    """Give a short colon-terminated header to the items under it that cannot stand alone.

    A header's scope is the blank-line block it opens, but a document extracted without a
    single blank line puts the whole text in one block, so scope alone is not a bound: the
    header is prepended only to a unit of at most `needy_words` words, which is a unit too
    thin to carry a topic by itself.  A full proposal keeps its own text and can never be
    labelled with a heading that has stopped applying.  A unit that opens with a bullet is
    an item, never a header, which keeps an item whose separator was read as a colon from
    swallowing the next one.  The header is emitted once on its own.
    """
    out, header = [], None
    for block in blocks:
        for s in block:
            if (s.endswith(":") and not LIST_LEAD_RE.match(s)
                    and len(WORD_RE.findall(s)) <= max_header_words):
                header = s[:-1].strip()
                out.append(s)
                continue
            if header is not None and len(WORD_RE.findall(s)) <= needy_words:
                out.append(f"{header}: {s}")
            else:
                out.append(s)
                header = None       # the run of items under the header has ended
        header = None
    return out


# ------------------------------------------------------------------ segmenter step
# A unit carries more than one sentence only if a full stop sits in its interior.
INTERNAL_TERM_RE = re.compile(r"[.!?][\"'”’\)\]]?\s+\S")
# A boundary is accepted only where the text before it ends in a full stop.
BOUND_OK_RE = re.compile(r"[.!?][\"'”’\)\]]?\s*$")
# ... and the word carrying that full stop, so a bound abbreviation can veto the boundary.
LAST_WORD_RE = re.compile(r"([^\W\d_]+)\.[\"'”’\)\]]?\s*$")


def needs_senter(unit):
    return bool(INTERNAL_TERM_RE.search(unit))


def ends_on_abbrev(text, abbrevs):
    """True when the full stop closing `text` belongs to a bound abbreviation."""
    m = LAST_WORD_RE.search(text)
    return bool(m) and m.group(1) in abbrevs


def split_at_stops(unit, nlp_doc_sents, abbrevs=frozenset()):
    """Keep the model boundaries that fall after a full stop, minus the abbreviations."""
    out = []
    for s in nlp_doc_sents:
        s = s.strip()
        if not s:
            continue
        prev = out[-1] if out else None
        if prev is not None and (not BOUND_OK_RE.search(prev)
                                 or ends_on_abbrev(prev, abbrevs)):
            out[-1] = f"{prev} {s}"
        else:
            out.append(s)
    return out or [unit]


def wc(s):
    return len(WORD_RE.findall(s))


def reattach_stranded(recs, max_short=SHORT_MAX):
    """Give a stranded tail back to the line it came from; leave every other unit alone."""
    out = []
    for text, meta in recs:
        if out and wc(text) <= max_short and _CONT_START_RE.match(text):
            ptext, pmeta = out[-1]
            out[-1] = (f"{ptext.rstrip()} {text.strip()}", pmeta)
            continue
        out.append((text, meta))
    return out


def enforce_min_words(recs, min_words=MIN_WORDS):
    """Merge every chunk under the word floor forward into its neighbour.

    `reattach_stranded` only recognises a lower-case continuation, so a fragment that
    does not start that way sails through untouched: a bare section-header label, a
    colon-list header `fix_colon_lists` failed to attach, a leaked table cell, or a
    punctuation-only fragment the trained segmenter carved out on its own (word_count 0
    - a lone '.', '...', a stray list marker). None of those carry a usable signal for
    the embedding, topic, or stance stages on their own, so none may stand alone as a
    chunk. A run of several straight is absorbed into one growing chunk; a trailing
    remainder with nothing left to absorb folds back into the chunk before it.
    """
    out = []
    for text, meta in recs:
        if out and wc(out[-1][0]) < min_words:
            out[-1] = (f"{out[-1][0].rstrip()} {text.strip()}", out[-1][1])
        else:
            out.append((text, meta))
    if len(out) >= 2 and wc(out[-1][0]) < min_words:
        out[-2] = (f"{out[-2][0].rstrip()} {out[-1][0].strip()}", out[-2][1])
        out.pop()
    return out


def stride_split(s, max_words=MAX_WORDS, overlap=STRIDE_OVERLAP):
    toks = s.split()
    counts = [wc(t) for t in toks]
    if sum(counts) <= max_words:
        return [s]
    out, i, n = [], 0, len(toks)
    while i < n:
        c, j = 0, i
        while j < n and (c + counts[j] <= max_words or j == i):
            c += counts[j]; j += 1
        out.append(" ".join(toks[i:j]))
        if j >= n:
            break
        back, k = 0, j
        while k > i + 1 and back < overlap:
            k -= 1; back += counts[k]
        i = k
    return out


# ------------------------------------------------------------------ model
def load_senter(model_name, lang=None):
    """Load tok2vec + senter only, with the abbreviation list the corpora actually need.

    Registering an abbreviation as a tokenizer special case keeps its full stop inside the
    token, so the model never sees a free-standing sentence-final period.  It does not
    forbid a boundary: sentence starts are marked per token, so a sentence really ending in
    "etc." can still be closed.
    """
    nlp = spacy.load(model_name)
    for p in list(nlp.pipe_names):
        if p not in ("tok2vec", "senter"):
            nlp.disable_pipe(p)
    if "senter" in nlp.component_names and "senter" not in nlp.pipe_names:
        nlp.enable_pipe("senter")
    nlp.max_length = SENTER_MAX_LEN
    for form in ABBREV_ALL.get(lang or nlp.lang, ()):
        nlp.tokenizer.add_special_case(f"{form}.", [{"ORTH": f"{form}."}])
    return nlp


## 4. Corpus loaders

In [6]:
def load_us():
    CSV_IN = ROOT / "data/us" / "policy_platforms.csv"
    KEY  = ["candidate_webname", "state_postal", "cd", "cand_party", "year"]
    META = ["policy_code", "issue_header", "year", "FECCandID", "BIOGUIDE_id", "statement_id"]
    def _sid(v):
        try:
            return (0, int(float(v)))
        except (TypeError, ValueError):
            return (1, str(v))
    docs = OrderedDict()
    with open(CSV_IN, newline="", encoding="utf-8") as f:
        for row in tqdm(csv.DictReader(f), desc="US load"):
            txt = (row.get("issue_text") or "").strip()
            if not txt:
                continue
            doc_id = "|".join(str(row.get(k, "")) for k in KEY)
            meta = {k: row.get(k, "") for k in META}
            docs.setdefault(doc_id, []).append(
                {"order": _sid(row.get("statement_id", "")),
                 "party_label": row.get("cand_party", ""), "meta": meta, "text": txt})
    return docs

def _s(v):
    # metadata fields are categorical labels -> store uniformly as str so the
    # column has one arrow type (matched rows carry native feather dtypes,
    # e.g. 'round' is int64; unmatched rows are filled blank).
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return str(v).strip()

def load_br():
    BRDIR = ROOT / "data" / "br"
    corpus = pd.read_feather(BRDIR / "br_manifestos_all.feather",
                             columns=["platform_id", "text_repaired"])
    mcols = ["platform_id", "party", "state", "municipality_name",
             "office", "round", "election_result"]
    pmap = pd.read_feather(BRDIR / "platform_party_map.feather", columns=mcols)
    pmap = pmap.drop_duplicates("platform_id").set_index("platform_id")
    mfields = ["state", "municipality_name", "office", "round", "election_result"]
    docs = OrderedDict()
    for pid, txt in tqdm(zip(corpus.platform_id, corpus.text_repaired),
                         total=len(corpus), desc="BR load"):
        txt = (txt or "").strip()
        if not txt:
            continue
        if pid in pmap.index:
            r = pmap.loc[pid]
            party = _s(r["party"]) or "UNK"
            meta = {k: _s(r[k]) for k in mfields}
        else:
            party, meta = "UNK", {k: "" for k in mfields}
        docs[pid] = [{"order": (0, 0), "party_label": party, "meta": meta, "text": txt}]
    return docs

# One cascade for both corpora: the only per-corpus settings left are the segmenter
# checkpoint and the batch size.
CORPORA = {
    "us": {"model": "en_core_web_lg", "batch": 128,
           "loader": load_us, "out": ROOT / "data/us" / "campaignview_chunks_sent"},
    "br": {"model": "pt_core_news_lg", "batch": 32,
           "loader": load_br, "out": ROOT / "data" / "br" / "br_manifestos_chunks_sent"},
}

## 5. Pipeline state &amp; report helper

In [ ]:
# Pipeline state + report helper. STATE caches each step's output per corpus, so a
# failure in a late step never forces re-running the expensive segmenter (Step 4).
STATE = {}             # corpus -> {n_docs, points, doc_points, texts, units, frags, df, ...}

def _report(corpus, n_docs, n_points, df, word_lens):
    word_lens.sort(); n = len(word_lens)
    def pct(p):
        if not word_lens:
            return 0
        k = (n - 1) * p / 100.0; f = int(k); c = min(f + 1, n - 1)
        return word_lens[f] + (word_lens[c] - word_lens[f]) * (k - f)
    print(f"\n  [{corpus.upper()}] chunks: {n:,} | docs: {df['doc_id'].nunique():,} | "
          f"mean chunks/doc: {n/max(1,df['doc_id'].nunique()):.1f}")
    print("  word-length percentiles:")
    for name, p in [("min",0),("p5",5),("p10",10),("p25",25),("p50",50),
                    ("p75",75),("p90",90),("p95",95),("max",100)]:
        print(f"    {name:<6}{pct(p):>8.0f}")
    print(f"    mean  {sum(word_lens)/max(1,n):>8.1f}")
    for thr in (2,3,4):
        c = sum(1 for w in word_lens if w < thr)
        print(f"    <{thr}w   {c:>8,} ({100*c/max(1,n):.2f}%)")
    for thr in (60,100,150):
        c = sum(1 for w in word_lens if w >= thr)
        print(f"    >={thr}w {c:>8,} ({100*c/max(1,n):.2f}%)")
    below_floor = sum(1 for w in word_lens if w < MIN_WORDS)
    print(f"  max words = {max(word_lens) if word_lens else 0} (cap {MAX_WORDS}); "
          f"below the {MIN_WORDS}-word floor: {below_floor:,} "
          f"(0 expected - a whole document shorter than {MIN_WORDS} words is the only exception)")
    print(f"  wrote {CORPORA[corpus]['out'].name}.feather (+ .csv)\n")

## 6. Run — single workflow, both corpora

Run Steps 1–6 top to bottom **once**. Every step processes US **and** BR in the same pass and caches its output into `STATE[corpus]`, so only **Step 4 (the segmenter)** is expensive and a re-run of Steps 5–6 never repeats it. Step 1 prints the metadata type census (the data issue that broke `to_feather`); Step 5 re-checks dtypes before writing.

In [8]:
# Step 0 - timing register + environment check.  Each step below records its own wall
# time per corpus into TIMING; the last cell prints the table and appends it to
# reports/sentencize_timing.md.  Run this before Step 1 and once per full run.
import time, platform
from datetime import datetime
import spacy.util

TIMING = {c: {} for c in ("us", "br")}
RUN_STARTED = datetime.now()

print("started :", RUN_STARTED.strftime("%Y-%m-%d %H:%M:%S"))
print("host    :", platform.node(), "|", os.cpu_count(), "cores | N_PROCESS =", N_PROCESS)
print("ROOT    :", ROOT)
for m in ("en_core_web_lg", "pt_core_news_lg"):
    print(f"{m:16s}:", "ok" if spacy.util.is_package(m) else "MISSING - run the install cell above")
for corpus in ("us", "br"):
    src = CORPORA[corpus]["loader"].__name__
    dst = CORPORA[corpus]["out"]
    print(f"{corpus:8s}: -> {dst.relative_to(ROOT)}.feather (+ .csv)")

started : 2026-08-13 00:46:49
host    : cc6995af4881 | 8 cores | N_PROCESS = 8
ROOT    : /content/drive/MyDrive/Papers/transfer_learning/topic2irt
en_core_web_lg  : ok
pt_core_news_lg : ok
us      : -> data/us/campaignview_chunks_sent.feather (+ .csv)
br      : -> data/br/br_manifestos_chunks_sent.feather (+ .csv)


In [9]:
# Step 1 - load BOTH corpora -> flat points, cache in STATE[corpus]. Fast.
# Prints the metadata type census per corpus: every field MUST be one python type
# (mixed int/str is what broke the feather write).
from collections import Counter
STATE.clear()
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    cfg = CORPORA[corpus]
    docs = cfg["loader"]()
    points, doc_points = [], OrderedDict()
    for doc_id, pts in docs.items():
        for p in sorted(pts, key=lambda x: x["order"]):
            doc_points.setdefault(doc_id, []).append(len(points))
            points.append(p)
    STATE[corpus] = {"n_docs": len(docs), "points": points, "doc_points": doc_points}
    metatypes = {}
    for p in points:
        for k, v in p["meta"].items():
            metatypes.setdefault(k, Counter())[type(v).__name__] += 1
    mixed = {k: dict(c) for k, c in metatypes.items() if len(c) > 1}
    TIMING[corpus]["1 load"] = time.perf_counter() - _t0
    print(f"[{corpus.upper()}] docs {len(docs):,} | points {len(points):,} | "
          f"mixed-type meta fields: {mixed or 'none'} | {TIMING[corpus]['1 load']:.1f}s")

US load: 0it [00:00, ?it/s]

[US] docs 4,507 | points 43,465 | mixed-type meta fields: none | 3.1s


BR load:   0%|          | 0/17385 [00:00<?, ?it/s]

[BR] docs 17,385 | points 17,385 | mixed-type meta fields: none | 11.6s


In [10]:
# Step 2 - normalize both corpora (ftfy mojibake repair -> NFKC -> missing-space fix)
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    S = STATE[corpus]
    S["texts"] = [normalize_text(p["text"]) for p in tqdm(S["points"], desc=f"{corpus} normalize")]
    TIMING[corpus]["2 normalize"] = time.perf_counter() - _t0
    print(f"[{corpus.upper()}] normalized {len(S['texts']):,} | {TIMING[corpus]['2 normalize']:.1f}s")

us normalize:   0%|          | 0/43465 [00:00<?, ?it/s]

[US] normalized 43,465 | 17.6s


br normalize:   0%|          | 0/17385 [00:00<?, ?it/s]

[BR] normalized 17,385 | 327.5s


In [11]:
# Step 3 - layout units, both corpora.  No model here: the page decides where a unit
# ends.  Itemized-list break, then the colon header goes to the items too short to carry
# a topic on their own.
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    S = STATE[corpus]
    units = [fix_colon_lists(split_units(t))
             for t in tqdm(S["texts"], desc=f"{corpus} units")]
    S["units"] = units
    TIMING[corpus]["3 units"] = time.perf_counter() - _t0
    print(f"[{corpus.upper()}] units: {sum(len(u) for u in units):,} | {TIMING[corpus]['3 units']:.1f}s")

us units:   0%|          | 0/43465 [00:00<?, ?it/s]

[US] units: 191,611 | 180.3s


br units:   0%|          | 0/17385 [00:00<?, ?it/s]

[BR] units: 2,774,477 | 1204.4s


In [12]:
# Step 4 - sentence-split the units that hold more than one sentence (the model step).
# A unit goes to the segmenter only if a full stop sits in its interior, and a boundary is
# kept only where it falls after a full stop that does not close a bound abbreviation.  So
# the model chooses which full stops end a sentence, can never open a break inside a
# clause, and can never cut a title or a legal reference off the word it introduces.
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    cfg = CORPORA[corpus]; S = STATE[corpus]
    nlp = load_senter(cfg["model"])
    abbrev = ABBREV_BOUND[nlp.lang]
    units = S["units"]
    todo = [(i, j) for i, u in enumerate(units) for j, x in enumerate(u) if needs_senter(x)]
    n_units = sum(len(u) for u in units)
    print(f"[{corpus.upper()}] units to segment: {len(todo):,} of {n_units:,} "
          f"| {len(abbrev):,} abbreviation forms guarded")
    out = [[[x] for x in u] for u in units]
    stream = nlp.pipe(((units[i][j], (i, j)) for i, j in todo), as_tuples=True,
                      n_process=N_PROCESS, batch_size=cfg["batch"])
    for doc, (i, j) in tqdm(stream, total=len(todo), desc=f"{corpus} senter"):
        out[i][j] = split_at_stops(units[i][j], [s.text for s in doc.sents], abbrev)
    S["frags"] = [[x for piece in u for x in piece] for u in out]
    TIMING[corpus]["4 senter"] = time.perf_counter() - _t0
    print(f"[{corpus.upper()}] fragments: {sum(len(f) for f in S['frags']):,} "
          f"| {TIMING[corpus]['4 senter']/60:.1f} min "
          f"({1000*TIMING[corpus]['4 senter']/max(1,len(todo)):.1f} ms per segmented unit)")

[US] units to segment: 105,579 of 191,611 | 263 abbreviation forms guarded


us senter:   0%|          | 0/105579 [00:00<?, ?it/s]

[US] fragments: 440,100 | 4.0 min (2.2 ms per segmented unit)
[BR] units to segment: 272,972 of 2,774,477 | 583 abbreviation forms guarded


br senter:   0%|          | 0/272972 [00:00<?, ?it/s]

[BR] fragments: 3,196,676 | 10.4 min (2.3 ms per segmented unit)


In [ ]:
# Step 5 - reattach stranded tails + word floor + 150-word cap -> build df, both corpora.
# Re-checks object-column types and flags any mixed-type column BEFORE the write.
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    S = STATE[corpus]
    points, doc_points, frags = S["points"], S["doc_points"], S["frags"]
    rows, word_lens = [], []
    for doc_id, pidxs in tqdm(doc_points.items(), total=S["n_docs"], desc=f"{corpus} reattach+cap"):
        recs = []
        party = points[pidxs[0]]["party_label"]
        for pi in pidxs:
            meta = points[pi]["meta"]
            for frag in frags[pi]:
                recs.append((frag, meta))
        if not recs:
            continue
        recs = enforce_min_words(reattach_stranded(recs))
        idx = 0
        for text, meta in recs:
            for chunk in stride_split(text):
                w = wc(chunk)
                row = {"chunk_id": f"{doc_id}#s{idx}", "doc_id": doc_id,
                       "party_label": party, "chunk_text": chunk, "word_count": w}
                row.update(meta); rows.append(row); word_lens.append(w); idx += 1
    df = pd.DataFrame(rows)
    S["df"], S["word_lens"] = df, word_lens
    bad = {c: sorted(df[c].map(lambda v: type(v).__name__).unique())
           for c in df.columns if df[c].dtype == object
           and df[c].map(lambda v: type(v).__name__).nunique() > 1}
    TIMING[corpus]["5 reattach+cap"] = time.perf_counter() - _t0
    print(f"[{corpus.upper()}] df {len(df):,} x {df.shape[1]} cols | "
          f"mixed-type object cols: {bad or 'none (safe to write)'} "
          f"| {TIMING[corpus]['5 reattach+cap']:.1f}s")

In [14]:
# Step 6 - write feather + csv and report, both corpora
for corpus in ("us", "br"):
    _t0 = time.perf_counter()
    S = STATE[corpus]; out = CORPORA[corpus]["out"]
    df = S["df"]
    df.to_feather(out.with_suffix(".feather"))
    df.to_csv(out.with_suffix(".csv"), index=False, encoding="utf-8")
    TIMING[corpus]["6 write"] = time.perf_counter() - _t0
    _report(corpus, S["n_docs"], len(S["points"]), df, list(S["word_lens"]))
    print(f"  write time: {TIMING[corpus]['6 write']:.1f}s\n")


  [US] chunks: 439,282 | docs: 4,507 | mean chunks/doc: 97.5
  word-length percentiles:
    min          0
    p5           4
    p10          6
    p25         11
    p50         18
    p75         25
    p90         33
    p95         39
    max        150
    mean      19.3
    <2w      3,081 (0.70%)
    <3w      9,471 (2.16%)
    <4w     17,081 (3.89%)
    >=60w    1,966 (0.45%)
    >=100w      119 (0.03%)
    >=150w       24 (0.01%)
  max words = 150 (cap 150); short chunks are headings and one-line list items, which now stand on their own
  wrote campaignview_chunks_sent.feather (+ .csv)

  write time: 12.1s


  [BR] chunks: 3,173,387 | docs: 17,385 | mean chunks/doc: 182.5
  word-length percentiles:
    min          0
    p5           2
    p10          4
    p25          8
    p50         15
    p75         24
    p90         35
    p95         44
    max        150
    mean      17.8
    <2w     87,118 (2.75%)
    <3w    171,219 (5.40%)
    <4w    243,148 (7.66%)
    >=60w   

In [15]:
# Step 7 - append this run to the pipeline timing register.  Every timed stage of the
# pipeline writes the same two files under reports/timing/ (a csv to combine stages with,
# and an md to read), and code/timing.py is their only writer.
import sys
sys.path.insert(0, str(ROOT / "code"))
from timing import log_run, CSV_PATH, MD_PATH

scale = {c: {"n_docs": STATE[c]["n_docs"], "n_chunks": len(STATE[c]["df"])}
         for c in ("us", "br")}
print(log_run("00 sentencize", TIMING, scale=scale, started=RUN_STARTED))
print("->", CSV_PATH)
print("->", MD_PATH)

## 00 sentencize · 2026-08-13 00:46:49 · cc6995af4881, 8 cores

| step | United States | Brazil |
|---|---:|---:|
| 1 load | 0:00:03 | 0:00:12 |
| 2 normalize | 0:00:18 | 0:05:28 |
| 3 units | 0:03:00 | 0:20:04 |
| 4 senter | 0:03:57 | 0:10:24 |
| 5 reattach+cap | 0:00:09 | 0:01:05 |
| 6 write | 0:00:12 | 0:00:27 |
| **total** | **0:07:40** | **0:37:39** |

| corpus | documents | chunks | seconds per 1k chunks |
|---|---:|---:|---:|
| United States | 4,507 | 439,282 | 1.0 |
| Brazil | 17,385 | 3,173,387 | 0.7 |

Wall clock over the whole stage: **0:45:19**.

-> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/reports/timing/pipeline_timing.csv
-> /content/drive/MyDrive/Papers/transfer_learning/topic2irt/reports/timing/pipeline_timing.md
